# 11 — Defensive AI and ML-Based Detection

## What This Is
ML-based detection systems learn normal behaviour patterns and flag anomalies. Applications include network intrusion detection (IDS), user/entity behaviour analytics (UEBA), log anomaly detection, and malware classification.

## Why It Matters
Signature-based detection (antivirus, IDS rules) misses novel attacks. ML-based systems detect unknown threats through behavioural deviation — the basis for next-gen EDR and SIEM platforms.

## Where the Community Stands
Isolation Forest and Autoencoder-based anomaly detection are workhorses for unsupervised detection. Supervised malware classification uses gradient boosted trees on PE header features. LLMs are being applied to log analysis (Security Copilot, Splunk AI).

In [ ]:
import random
import math
import statistics
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

# Simulate network flow features
def generate_normal_flow(rng: random.Random) -> Dict:
    return {
        'bytes_out':     rng.gauss(50000, 15000),
        'bytes_in':      rng.gauss(200000, 60000),
        'duration_sec':  rng.gauss(30, 10),
        'packet_count':  rng.gauss(150, 40),
        'unique_ports':  rng.randint(1, 5),
        'hour_of_day':   rng.randint(8, 18),   # business hours
        'label':         'normal'
    }

def generate_attack_flow(rng: random.Random, attack_type: str) -> Dict:
    if attack_type == 'exfiltration':
        return {
            'bytes_out':    rng.gauss(5000000, 500000),  # large upload
            'bytes_in':     rng.gauss(5000, 2000),
            'duration_sec': rng.gauss(300, 50),
            'packet_count': rng.gauss(3000, 500),
            'unique_ports':  1,
            'hour_of_day':  rng.randint(0, 6),           # off-hours
            'label':        'exfiltration'
        }
    elif attack_type == 'portscan':
        return {
            'bytes_out':    rng.gauss(500, 100),
            'bytes_in':     rng.gauss(1000, 200),
            'duration_sec': rng.gauss(5, 2),
            'packet_count': rng.gauss(500, 100),
            'unique_ports': rng.randint(50, 1024),        # many ports
            'hour_of_day':  rng.randint(0, 23),
            'label':        'portscan'
        }
    else:  # C2 beacon
        return {
            'bytes_out':    rng.gauss(1000, 200),
            'bytes_in':     rng.gauss(500, 100),
            'duration_sec': rng.gauss(600, 30),           # long session
            'packet_count': rng.gauss(200, 30),
            'unique_ports':  1,
            'hour_of_day':  rng.randint(0, 23),
            'label':        'c2_beacon'
        }

rng = random.Random(42)
dataset = []
for _ in range(800):
    dataset.append(generate_normal_flow(rng))
for _ in range(80):
    dataset.append(generate_attack_flow(rng, 'exfiltration'))
for _ in range(60):
    dataset.append(generate_attack_flow(rng, 'portscan'))
for _ in range(60):
    dataset.append(generate_attack_flow(rng, 'c2_beacon'))

FEATURES = ['bytes_out', 'bytes_in', 'duration_sec', 'packet_count', 'unique_ports', 'hour_of_day']
rng.shuffle(dataset)
print(f'Dataset: {len(dataset)} flows  ({sum(1 for d in dataset if d["label"]=="normal")} normal, {sum(1 for d in dataset if d["label"]!="normal")} attacks)')

In [ ]:
class ZScoreAnomalyDetector:
    """Univariate z-score anomaly detector per feature."""

    def __init__(self, threshold: float = 3.0):
        self.threshold = threshold
        self.means: Dict[str, float] = {}
        self.stds:  Dict[str, float] = {}

    def fit(self, data: List[Dict]) -> None:
        for feat in FEATURES:
            vals = [d[feat] for d in data]
            self.means[feat] = statistics.mean(vals)
            self.stds[feat]  = statistics.stdev(vals) or 1e-8

    def score(self, sample: Dict) -> float:
        z_scores = []
        for feat in FEATURES:
            z = abs(sample[feat] - self.means[feat]) / self.stds[feat]
            z_scores.append(z)
        return max(z_scores)  # worst-case z

    def predict(self, sample: Dict) -> str:
        return 'anomaly' if self.score(sample) > self.threshold else 'normal'

# Train on normal data only (unsupervised)
train_normal = [d for d in dataset if d['label'] == 'normal'][:600]
detector = ZScoreAnomalyDetector(threshold=3.5)
detector.fit(train_normal)

# Evaluate
tp = fp = tn = fn = 0
for sample in dataset:
    pred  = detector.predict(sample)
    truth = 'anomaly' if sample['label'] != 'normal' else 'normal'
    if pred == 'anomaly' and truth == 'anomaly': tp += 1
    elif pred == 'anomaly' and truth == 'normal': fp += 1
    elif pred == 'normal' and truth == 'normal': tn += 1
    else: fn += 1

precision = tp / (tp + fp) if (tp+fp) > 0 else 0
recall    = tp / (tp + fn) if (tp+fn) > 0 else 0
f1        = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0

print('Z-Score Anomaly Detector:')
print(f'  TP={tp} FP={fp} TN={tn} FN={fn}')
print(f'  Precision: {precision:.3f}  Recall: {recall:.3f}  F1: {f1:.3f}')

## Isolation Forest (Simulated)
Isolation Forest isolates anomalies by randomly partitioning features. Anomalies require fewer splits to isolate (shorter path length) because they occupy sparse regions of feature space.

In [ ]:
class IsolationTree:
    """Single isolation tree."""

    def __init__(self, max_depth: int = 8):
        self.max_depth = max_depth
        self.split_feature: Optional[str] = None
        self.split_value:   Optional[float] = None
        self.left:          Optional['IsolationTree'] = None
        self.right:         Optional['IsolationTree'] = None
        self.size:          int = 0

    def fit(self, data: List[Dict], depth: int = 0) -> None:
        self.size = len(data)
        if depth >= self.max_depth or len(data) <= 1:
            return
        feat = random.choice(FEATURES)
        vals = [d[feat] for d in data]
        lo, hi = min(vals), max(vals)
        if lo >= hi:
            return
        split = random.uniform(lo, hi)
        left_data  = [d for d in data if d[feat] < split]
        right_data = [d for d in data if d[feat] >= split]
        if not left_data or not right_data:
            return
        self.split_feature = feat
        self.split_value   = split
        self.left  = IsolationTree(self.max_depth)
        self.right = IsolationTree(self.max_depth)
        self.left.fit(left_data,  depth+1)
        self.right.fit(right_data, depth+1)

    def path_length(self, sample: Dict, depth: int = 0) -> float:
        if self.split_feature is None:
            n = self.size
            # Expected path length correction for leaf
            if n <= 1: return depth
            return depth + 2*(math.log(n-1) + 0.5772) - 2*(n-1)/n
        if sample[self.split_feature] < self.split_value:
            return self.left.path_length(sample, depth+1)
        return self.right.path_length(sample, depth+1)

class IsolationForest:
    def __init__(self, n_trees: int = 50, max_depth: int = 8):
        self.trees: List[IsolationTree] = []
        self.n_trees   = n_trees
        self.max_depth = max_depth
        self.avg_path: float = 0.0

    def fit(self, data: List[Dict], subsample: int = 256) -> None:
        for _ in range(self.n_trees):
            sample = random.sample(data, min(subsample, len(data)))
            tree   = IsolationTree(self.max_depth)
            tree.fit(sample)
            self.trees.append(tree)
        n = len(data)
        self.avg_path = 2*(math.log(n-1)+0.5772) - 2*(n-1)/n if n > 1 else 1.0

    def anomaly_score(self, sample: Dict) -> float:
        avg_len = sum(t.path_length(sample) for t in self.trees) / len(self.trees)
        return 2 ** (-avg_len / self.avg_path)

    def predict(self, sample: Dict, threshold: float = 0.6) -> str:
        return 'anomaly' if self.anomaly_score(sample) > threshold else 'normal'

iforest = IsolationForest(n_trees=30, max_depth=6)
iforest.fit(train_normal)

tp2 = fp2 = tn2 = fn2 = 0
for sample in dataset:
    pred  = iforest.predict(sample, threshold=0.58)
    truth = 'anomaly' if sample['label'] != 'normal' else 'normal'
    if pred == 'anomaly' and truth == 'anomaly': tp2 += 1
    elif pred == 'anomaly' and truth == 'normal': fp2 += 1
    elif pred == 'normal' and truth == 'normal': tn2 += 1
    else: fn2 += 1

p2 = tp2/(tp2+fp2) if (tp2+fp2)>0 else 0
r2 = tp2/(tp2+fn2) if (tp2+fn2)>0 else 0
f2 = 2*p2*r2/(p2+r2) if (p2+r2)>0 else 0
print('Isolation Forest:')
print(f'  TP={tp2} FP={fp2} TN={tn2} FN={fn2}')
print(f'  Precision: {p2:.3f}  Recall: {r2:.3f}  F1: {f2:.3f}')

## Log Anomaly Detection with n-gram Fingerprinting
Log anomaly detection builds a vocabulary of normal log templates. New log lines matching no known template trigger alerts. This catches novel malware activity that leaves unfamiliar log patterns.

In [ ]:
from collections import Counter

NORMAL_LOG_TEMPLATES = [
    'User {user} logged in from {ip}',
    'File {file} opened by {user}',
    'Service {svc} started successfully',
    'Network connection to {ip}:{port} established',
    'Backup completed: {bytes} bytes written',
    'Authentication successful for {user}',
    'DNS query for {domain} resolved',
    'HTTP GET {path} returned 200',
]

ATTACK_LOGS = [
    'powershell.exe -encodedCommand JABjAD...',
    'Process lsass.exe accessed by mimikatz.exe',
    'Registry key HKLM\\Run modified: malware.exe',
    'Outbound connection to 203.0.113.42:4444',
    'cmd.exe spawned by word.exe',
    'whoami /priv executed by user guest',
]

def tokenize_log(line: str) -> List[str]:
    # Remove variable parts (IPs, usernames, numbers)
    line = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '<IP>', line) if 'import re' in dir() else line
    line = re.sub(r'\b\d+\b', '<NUM>', line)
    return line.lower().split()

import re

def build_normal_vocab(templates: List[str]) -> Counter:
    bigrams = Counter()
    for t in templates:
        tokens = tokenize_log(t)
        for i in range(len(tokens)-1):
            bigrams[(tokens[i], tokens[i+1])] += 1
    return bigrams

def log_anomaly_score(line: str, normal_vocab: Counter) -> float:
    tokens  = tokenize_log(line)
    if len(tokens) < 2:
        return 0.0
    total   = len(tokens) - 1
    unknown = sum(1 for i in range(total)
                  if (tokens[i], tokens[i+1]) not in normal_vocab)
    return unknown / total

vocab = build_normal_vocab(NORMAL_LOG_TEMPLATES)

print('Log anomaly detection (bigram novelty):\n')
print(f'{"Log line":<55} {"Anomaly%"}')
print('-' * 70)

test_logs = [
    'User alice logged in from 10.0.1.5',
    'HTTP GET /api/health returned 200',
] + ATTACK_LOGS

for log in test_logs:
    score = log_anomaly_score(log, vocab)
    flag  = '  *** ALERT ***' if score > 0.6 else ''
    print(f'{log[:55]:<55} {score:.2f}{flag}')